In [2]:
%pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

import os
import pandas as pd

# Assuming the dataset is downloaded to 'path' and it contains a CSV file
# Adjust the filename based on the actual downloaded dataset structure
dataset_file = os.path.join(path, 'IMDB Dataset.csv')

# Read the dataset
df = pd.read_csv(dataset_file)

# Separate positive and negative reviews
positive_reviews = df[df['sentiment'] == 'positive']
negative_reviews = df[df['sentiment'] == 'negative']

# Determine the number of samples to take for each sentiment
# We want 5000 samples in total, split between positive and negative
samples_per_sentiment = 500 // 2

# Sample from each sentiment
positive_sample = positive_reviews.sample(n=samples_per_sentiment, random_state=42) # Use a random_state for reproducibility
negative_sample = negative_reviews.sample(n=samples_per_sentiment, random_state=42)

# Concatenate the samples to create the final dataset of 500 samples
initial_sample_df = pd.concat([positive_sample, negative_sample])

# Shuffle the resulting DataFrame to mix positive and negative reviews
initial_sample_df = initial_sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Initial sample of 500 data points created.")
print(initial_sample_df.head())
print(initial_sample_df['sentiment'].value_counts())


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Diego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


100%|██████████| 25.7M/25.7M [00:02<00:00, 11.2MB/s]

Extracting files...


Path to dataset files: C:\Users\Diego\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1
Initial sample of 500 data points created.
                                              review sentiment
0  WARNING:I advise anyone who has not seen the f...  negative
1  Always fancied this film from the video cover....  positive
2  Jerry spies Tom listening to a creepy story on...  negative
3  Following my experience of Finland for slightl...  positive
4  <br /><br />This film has some really impressi...  positive
sentiment
negative    250
positive    250
Name: count, dtype: int64


In [3]:
import re

def remove_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

initial_sample_df['review_cleaned'] = initial_sample_df['review'].apply(remove_html)

print("\nDataFrame with HTML tags removed:")
print(initial_sample_df.head())


DataFrame with HTML tags removed:
                                              review sentiment  \
0  WARNING:I advise anyone who has not seen the f...  negative   
1  Always fancied this film from the video cover....  positive   
2  Jerry spies Tom listening to a creepy story on...  negative   
3  Following my experience of Finland for slightl...  positive   
4  <br /><br />This film has some really impressi...  positive   

                                      review_cleaned  
0  WARNING:I advise anyone who has not seen the f...  
1  Always fancied this film from the video cover....  
2  Jerry spies Tom listening to a creepy story on...  
3  Following my experience of Finland for slightl...  
4  This film has some really impressive action sc...  


In [4]:

initial_sample_df['review_cleaned'] = initial_sample_df['review_cleaned'].str.lower()

print("\nDataFrame with text converted to lowercase:")
print(initial_sample_df.head())


DataFrame with text converted to lowercase:
                                              review sentiment  \
0  WARNING:I advise anyone who has not seen the f...  negative   
1  Always fancied this film from the video cover....  positive   
2  Jerry spies Tom listening to a creepy story on...  negative   
3  Following my experience of Finland for slightl...  positive   
4  <br /><br />This film has some really impressi...  positive   

                                      review_cleaned  
0  warning:i advise anyone who has not seen the f...  
1  always fancied this film from the video cover....  
2  jerry spies tom listening to a creepy story on...  
3  following my experience of finland for slightl...  
4  this film has some really impressive action sc...  


Preprocesamiento óptimo con SpaCy para análisis de sentimientos
Configuración fundamental de SpaCy
La investigación reciente demuestra que preservar la estructura lingüística es más importante que la limpieza agresiva para el análisis de sentimientos. ScienceDirect SpaCy ofrece el equilibrio ideal entre procesamiento sofisticado y preservación del contexto semántico.

In [6]:
%pip install spacy
%pip install -U spacy
!python -m spacy download en_core_web_sm

import spacy
from spacy.lang.en.stop_words import STOP_WORDS

# Configuración óptima para análisis de sentimientos
nlp = spacy.load("en_core_web_sm")

class OptimizedIMDBPreprocessor:
    def __init__(self):
        # Desactivar componentes innecesarios para mejor rendimiento
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
        
    def preprocess_text(self, text):
        doc = self.nlp(text)
        processed_tokens = []
        
        for token in doc:
            # Saltar espacios en blanco y tokens puramente numéricos
            if token.is_space or token.text.isdigit():
                continue
                
            # Mantener puntuación importante para sentimientos
            if token.is_punct and token.text not in ["!", "?", ".", "..."]:
                continue
                
            # Usar forma lematizada en minúsculas
            processed_tokens.append(token.lemma_.lower())
        
        return " ".join(processed_tokens)

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---------- ----------------------------- 3.9/14.9 MB 18.1 MB/s eta 0:00:01
   ----------------------- ---------------- 8.7/14.9 MB 20.7 MB/s eta 0:00:01
   ----------------------------------- ---- 13.4/14.9 MB 21.5 MB/s eta 0:00:01
   ---------------------------------------- 14.9/14.9 MB 18.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/632.6 kB ? eta -:--:--
   --------------------------------------- 632.6/632.6 kB 11.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 19.4 MB/s eta 0:00:00
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
   -------------------------- ------------- 4.2/6.2 MB 19.4 MB/s eta 0:00:01
   ---------------------------------------- 6.2/6.2 MB 15.


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Diego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Diego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.6/12.8 MB 9.3 MB/s eta 0:00:02
     ------------ --------------------------- 3.9/12.8 MB 10.2 MB/s eta 0:00:01
     ------------------ --------------------- 6.0/12.8 MB 10.2 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 10.3 MB/s eta 0:00:01
     -------------------------------- ------ 10.7/12.8 MB 10.6 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 10.6 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 9.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Diego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Tratamiento crítico de stopwords y negaciones
Hallazgo clave de 2024: Las stopwords NO deben eliminarse en análisis de sentimientos. Las palabras como "not", "never", "no" son fundamentales para determinar la polaridad.

In [8]:
def handle_negation_patterns(doc):
    """Manejo avanzado de negaciones preservando el contexto"""
    negation_tokens = ["not", "no", "never", "nobody", "nothing", "neither", 
                      "nowhere", "none", "hardly", "scarcely", "barely", "n't"]
    
    tokens = []
    negate = False
    
    for token in doc:
        if token.text.lower() in negation_tokens:
            negate = True
            tokens.append(token.lemma_)
        elif token.is_punct and token.text in [".", "!", "?"]:
            negate = False
            tokens.append(token.text)
        elif negate and token.pos_ in ["ADJ", "VERB", "ADV"]:
            # Marcar palabras negadas para preservar contexto
            tokens.append(f"NOT_{token.lemma_}")
        else:
            tokens.append(token.lemma_)
    
    return tokens

Tratamiento de emoticones y caracteres especiales
Los estudios de 2024 confirman que los emoticones mejoran significativamente la precisión del análisis de sentimientos. Más del 50% de comentarios en plataformas digitales contienen emojis que proporcionan señales de sentimiento muy valiosas.

In [11]:
!pip install emoji
import emoji

def preserve_sentiment_signals(text):
    """Preservar emoticones y señales de sentimiento"""
    # Opción 1: Mantener emojis como tokens (recomendado)
    # SpaCy maneja bien los emoticones comunes
    
    # Opción 2: Convertir a descripciones textuales si es necesario
    text_with_descriptions = emoji.demojize(text)
    
    # Preservar patrones de puntuación que expresan sentimiento
    # Mantener: !, ?, ..., múltiples signos de exclamación
    # Remover: @, #, URLs (menos relevantes para sentimiento)
    
    return text_with_descriptions

   ---------------------------------------- 0.0/590.6 kB ? eta -:--:--
   ---------------------------------------- 590.6/590.6 kB 7.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Diego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Implementación optimizada de TF-IDF

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Configuración óptima basada en investigación 2024-2025
optimal_tfidf_params = {
    'ngram_range': (1, 2),  # Unigramas + bigramas para contexto
    'max_features': 20000,  # Balance entre rendimiento y memoria
    'min_df': 2,  # Remover palabras muy raras
    'max_df': 0.7,  # Remover palabras muy comunes
    'sublinear_tf': True,  # Escalado logarítmico de frecuencias
    'norm': 'l2',  # Normalización L2
    'smooth_idf': True,  # Smoothing de pesos IDF
    'stop_words': None,  # NO remover stopwords para sentimientos
    'strip_accents': 'unicode'
}

def create_hybrid_tfidf_features(train_texts, test_texts):
    """TF-IDF híbrido combinando palabras y caracteres"""
    # Vectorizador de palabras
    word_vectorizer = TfidfVectorizer(
        analyzer='word',
        **optimal_tfidf_params
    )
    
    # Vectorizador de caracteres (para capturar patrones ortográficos)
    char_vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=(1, 3),
        max_features=10000,
        sublinear_tf=True
    )
    
    # Combinar características
    word_features_train = word_vectorizer.fit_transform(train_texts)
    char_features_train = char_vectorizer.fit_transform(train_texts)
    
    combined_train = hstack([word_features_train, char_features_train])
    
    # Aplicar a datos de prueba
    word_features_test = word_vectorizer.transform(test_texts)
    char_features_test = char_vectorizer.transform(test_texts)
    combined_test = hstack([word_features_test, char_features_test])
    
    return combined_train, combined_test, word_vectorizer, char_vectorizer

Optimización de vocabulario y dimensionalidad
La investigación reciente establece que 15,000-20,000 características ofrecen el mejor balance entre rendimiento y eficiencia computacional: 

In [13]:
from sklearn.feature_selection import SelectKBest, chi2

def optimize_feature_selection(X_train, y_train, X_test, k=15000):
    """Selección optimizada de características"""
    selector = SelectKBest(chi2, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_test_selected = selector.transform(X_test)
    
    # Obtener características más importantes
    feature_scores = selector.scores_
    selected_features = selector.get_support(indices=True)
    
    return X_train_selected, X_test_selected, selector, feature_scores

Arquitecturas de redes neuronales densas
Las redes neuronales densas alcanzan 85-88% de precisión con la arquitectura correcta:

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

def create_optimized_dense_network(input_dim, num_classes=2):
    """Red neuronal densa optimizada para análisis de sentimientos"""
    model = Sequential([
        # Capa de entrada con normalización
        Dense(512, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.5),
        
        # Capas ocultas con arquitectura piramidal
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(128, activation='relu'),
        Dropout(0.2),
        
        Dense(64, activation='relu'),
        
        # Capa de salida
        Dense(1, activation='sigmoid')
    ])
    
    # Compilación optimizada
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    
    return model

Redes LSTM para comprensión secuencial
Las LSTM bidireccionales logran 87% de precisión capturando dependencias temporales:

In [15]:
from tensorflow.keras.layers import LSTM, Bidirectional, Embedding

def create_lstm_sentiment_model(vocab_size, embedding_dim=128, lstm_units=64):
    """Modelo LSTM bidireccional para análisis de sentimientos"""
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=512),
        
        # LSTM bidireccional con dropout
        Bidirectional(LSTM(
            lstm_units, 
            dropout=0.5, 
            recurrent_dropout=0.5,
            return_sequences=False
        )),
        
        # Capas densas finales
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [16]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

def comprehensive_cross_validation(model, X, y, cv_folds=5):
    """Validación cruzada exhaustiva con múltiples métricas"""
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }
    
    # K-Fold estratificado para mantener distribución de clases
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    
    # Validación cruzada con métricas múltiples
    cv_results = cross_validate(
        model, X, y, 
        cv=cv, 
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )
    
    # Calcular estadísticas
    results_summary = {}
    for metric in scoring.keys():
        test_scores = cv_results[f'test_{metric}']
        train_scores = cv_results[f'train_{metric}']
        
        results_summary[metric] = {
            'test_mean': np.mean(test_scores),
            'test_std': np.std(test_scores),
            'train_mean': np.mean(train_scores),
            'train_std': np.std(train_scores),
            'overfitting': np.mean(train_scores) - np.mean(test_scores)
        }
    
    return results_summary, cv_results

In [17]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

def comprehensive_sentiment_evaluation(y_true, y_pred, y_proba=None):
    """Evaluación exhaustiva para análisis de sentimientos binario"""
    
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }
    
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    
    # Matriz de confusión
    cm = confusion_matrix(y_true, y_pred)
    
    # Crear visualización
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Matriz de confusión
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Negativo', 'Positivo'],
                yticklabels=['Negativo', 'Positivo'], ax=axes[0])
    axes[0].set_title('Matriz de Confusión')
    axes[0].set_xlabel('Predicción')
    axes[0].set_ylabel('Realidad')
    
    # Gráfico de barras de métricas
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())
    
    bars = axes[1].bar(metric_names, metric_values, color='steelblue', alpha=0.7)
    axes[1].set_title('Métricas de Rendimiento')
    axes[1].set_ylabel('Puntuación')
    axes[1].set_ylim(0, 1)
    
    # Añadir valores sobre las barras
    for bar, value in zip(bars, metric_values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{value:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return metrics, cm

In [21]:
from sklearn.model_selection import learning_curve, validation_curve

def plot_comprehensive_learning_analysis(estimator, X, y, param_name='C', 
                                       param_range=np.logspace(-4, 2, 10)):
    """Análisis completo de aprendizaje y validación"""
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Curvas de aprendizaje
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='accuracy', n_jobs=-1,
        return_times=False
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    axes[0].plot(train_sizes, train_mean, 'o-', color='blue', label='Entrenamiento')
    axes[0].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                        alpha=0.1, color='blue')
    
    axes[0].plot(train_sizes, val_mean, 'o-', color='red', label='Validación')
    axes[0].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                        alpha=0.1, color='red')
    
    axes[0].set_xlabel('Tamaño del conjunto de entrenamiento')
    axes[0].set_ylabel('Precisión')
    axes[0].set_title('Curvas de Aprendizaje')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Curvas de validación
    train_scores_val, val_scores_val = validation_curve(
        estimator, X, y, param_name=param_name, param_range=param_range,
        cv=5, scoring='accuracy', n_jobs=-1
    )
    
    train_mean_val = np.mean(train_scores_val, axis=1)
    train_std_val = np.std(train_scores_val, axis=1)
    val_mean_val = np.mean(val_scores_val, axis=1)
    val_std_val = np.std(val_scores_val, axis=1)
    
    axes[1].semilogx(param_range, train_mean_val, 'o-', color='blue', label='Entrenamiento')
    axes[1].fill_between(param_range, train_mean_val - train_std_val, 
                        train_mean_val + train_std_val, alpha=0.1, color='blue')
    
    axes[1].semilogx(param_range, val_mean_val, 'o-', color='red', label='Validación')
    axes[1].fill_between(param_range, val_mean_val - val_std_val, 
                        val_mean_val + val_std_val, alpha=0.1, color='red')
    
    axes[1].set_xlabel(param_name)
    axes[1].set_ylabel('Precisión')
    axes[1].set_title('Curvas de Validación')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [23]:
%pip install plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import spacy
import re
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuración de estilo
plt.style.use('default')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

class IMDBSentimentAnalysisVisualizer:
    def __init__(self, dataset_path=None):
        self.dataset_path = dataset_path
        self.nlp = None
        self.vectorizers = {}
        self.models = {}
        self.results = {}
        
    def load_and_preprocess_data(self, sample_size=None):
        """Cargar y preprocesar el dataset IMDB"""
        
        # Cargar dataset (ejemplo con datasets de sklearn o cargar CSV)
        try:
            # Opción 1: Cargar desde archivo CSV
            if self.dataset_path:
                df = pd.read_csv(self.dataset_path)
                texts = df['review'].values
                labels = df['sentiment'].map({'positive': 1, 'negative': 0}).values
            else:
                # Opción 2: Usar dataset de ejemplo
                from sklearn.datasets import fetch_20newsgroups
                print("Usando dataset de ejemplo (20newsgroups adaptado)...")
                categories = ['alt.atheism', 'soc.religion.christian']
                newsgroups = fetch_20newsgroups(subset='all', categories=categories, 
                                              remove=('headers', 'footers', 'quotes'))
                texts = newsgroups.data
                labels = newsgroups.target
                
        except Exception as e:
            print(f"Error cargando datos: {e}")
            # Generar datos sintéticos para demostración
            print("Generando datos sintéticos para demostración...")
            texts, labels = self._generate_synthetic_imdb_data(sample_size or 5000)
        
        if sample_size and len(texts) > sample_size:
            indices = np.random.choice(len(texts), sample_size, replace=False)
            texts = [texts[i] for i in indices]
            labels = labels[indices]
        
        return texts, labels
    
    def _generate_synthetic_imdb_data(self, n_samples=5000):
        """Generar datos sintéticos que simulan reseñas IMDB"""
        
        positive_words = ['excellent', 'amazing', 'outstanding', 'brilliant', 'fantastic', 
                         'wonderful', 'perfect', 'great', 'awesome', 'superb', 'incredible',
                         'magnificent', 'spectacular', 'marvelous', 'exceptional']
        
        negative_words = ['terrible', 'awful', 'horrible', 'disgusting', 'disappointing',
                         'boring', 'waste', 'worst', 'pathetic', 'ridiculous', 'annoying',
                         'frustrating', 'uninteresting', 'poorly', 'badly']
        
        neutral_words = ['movie', 'film', 'story', 'character', 'plot', 'scene', 'acting',
                        'director', 'cinema', 'watch', 'time', 'people', 'way', 'good', 'bad']
        
        texts = []
        labels = []
        
        for i in range(n_samples):
            # Generar reseña positiva o negativa
            is_positive = np.random.choice([0, 1])
            
            if is_positive:
                sentiment_words = positive_words
                label = 1
            else:
                sentiment_words = negative_words
                label = 0
            
            # Generar texto
            length = np.random.randint(20, 100)
            review_words = []
            
            for _ in range(length):
                if np.random.random() < 0.3:  # 30% sentiment words
                    review_words.append(np.random.choice(sentiment_words))
                else:
                    review_words.append(np.random.choice(neutral_words))
            
            texts.append(' '.join(review_words))
            labels.append(label)
        
        return texts, np.array(labels)
    
    def preprocess_with_spacy(self, texts):
        """Preprocesamiento con SpaCy"""
        
        if self.nlp is None:
            try:
                self.nlp = spacy.load("en_core_web_sm")
            except OSError:
                print("Modelo SpaCy no encontrado. Usando preprocesamiento básico.")
                return [self._basic_preprocess(text) for text in texts]
        
        processed_texts = []
        
        for text in texts:
            doc = self.nlp(text)
            tokens = []
            
            for token in doc:
                if not token.is_space and not token.is_punct and len(token.text) > 2:
                    tokens.append(token.lemma_.lower())
            
            processed_texts.append(' '.join(tokens))
        
        return processed_texts
    
    def _basic_preprocess(self, text):
        """Preprocesamiento básico sin SpaCy"""
        text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
        words = text.split()
        return ' '.join([word for word in words if len(word) > 2])
    
    def create_vectorization_comparison(self, texts, labels):
        """Comparar diferentes métodos de vectorización"""
        
        # Dividir datos
        X_train_text, X_test_text, y_train, y_test = train_test_split(
            texts, labels, test_size=0.2, stratify=labels, random_state=42
        )
        
        # Métodos de vectorización
        vectorization_methods = {
            'Multi-hot Encoding': CountVectorizer(binary=True, max_features=10000, ngram_range=(1,1)),
            'Bag of Words': CountVectorizer(max_features=10000, ngram_range=(1,1)),
            'TF-IDF (1-gram)': TfidfVectorizer(max_features=10000, ngram_range=(1,1)),
            'TF-IDF (1-2gram)': TfidfVectorizer(max_features=15000, ngram_range=(1,2))
        }
        
        # Evaluar cada método con SVM Linear
        vectorization_results = {}
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()
        
        for idx, (method_name, vectorizer) in enumerate(vectorization_methods.items()):
            print(f"Evaluando {method_name}...")
            
            # Vectorizar
            X_train_vec = vectorizer.fit_transform(X_train_text)
            X_test_vec = vectorizer.transform(X_test_text)
            
            # Modelo SVM
            svm = LinearSVC(C=1.0, random_state=42, max_iter=10000)
            
            # Curvas de aprendizaje
            train_sizes, train_scores, val_scores = learning_curve(
                svm, X_train_vec, y_train,
                train_sizes=np.linspace(0.1, 1.0, 10),
                cv=5, scoring='accuracy', n_jobs=-1
            )
            
            train_mean = np.mean(train_scores, axis=1)
            train_std = np.std(train_scores, axis=1)
            val_mean = np.mean(val_scores, axis=1)
            val_std = np.std(val_scores, axis=1)
            
            # Graficar
            axes[idx].plot(train_sizes, train_mean, 'o-', color='#1f77b4', 
                          label='Entrenamiento', linewidth=2, markersize=6)
            axes[idx].fill_between(train_sizes, train_mean - train_std, 
                                  train_mean + train_std, alpha=0.2, color='#1f77b4')
            
            axes[idx].plot(train_sizes, val_mean, 'o-', color='#ff7f0e', 
                          label='Validación', linewidth=2, markersize=6)
            axes[idx].fill_between(train_sizes, val_mean - val_std, 
                                  val_mean + val_std, alpha=0.2, color='#ff7f0e')
            
            axes[idx].set_xlabel('Tamaño del conjunto de entrenamiento')
            axes[idx].set_ylabel('Precisión')
            axes[idx].set_title(f'{method_name}', fontsize=14, fontweight='bold')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)
            
            # Añadir información
            final_score = val_mean[-1]
            axes[idx].text(0.02, 0.95, f'Precisión final: {final_score:.3f}', 
                          transform=axes[idx].transAxes, fontsize=10,
                          bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
            
            vectorization_results[method_name] = {
                'final_accuracy': final_score,
                'max_accuracy': np.max(val_mean),
                'vectorizer': vectorizer
            }
        
        plt.tight_layout()
        plt.suptitle('Comparación de Métodos de Vectorización (con SVM Linear)', 
                     fontsize=16, fontweight='bold', y=1.02)
        plt.savefig('vectorization_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        return vectorization_results, X_train_vec, X_test_vec, y_train, y_test
    
    def detailed_model_analysis(self, X_train, X_test, y_train, y_test):
        """Análisis detallado por modelo con múltiples métricas"""
        
        models = {
            'SVM Linear': LinearSVC(C=1.0, random_state=42, max_iter=10000, dual=False),
            'Logistic Regression': LogisticRegression(C=1.0, solver='liblinear', random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42),
            'Naive Bayes': MultinomialNB(alpha=0.1)
        }
        
        # Crear dashboard de análisis
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=['Curvas de Aprendizaje', 'Métricas de Validación Cruzada', 
                           'Tiempo de Entrenamiento', 'Matriz de Confusión Promedio'],
            specs=[[{"secondary_y": False}, {"type": "bar"}],
                   [{"type": "bar"}, {"type": "heatmap"}]]
        )
        
        detailed_results = {}
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
        
        for idx, (name, model) in enumerate(models.items()):
            print(f"Analizando {name}...")
            
            # Curvas de aprendizaje
            train_sizes, train_scores, val_scores = learning_curve(
                model, X_train, y_train,
                train_sizes=np.linspace(0.1, 1.0, 8),
                cv=5, scoring='accuracy', n_jobs=-1
            )
            
            val_mean = np.mean(val_scores, axis=1)
            
            # Añadir curva de aprendizaje
            fig.add_trace(
                go.Scatter(x=train_sizes, y=val_mean, mode='lines+markers',
                          name=f'{name}', line=dict(color=colors[idx], width=2)),
                row=1, col=1
            )
            
            # Validación cruzada detallada
            from sklearn.model_selection import cross_validate
            cv_results = cross_validate(
                model, X_train, y_train,
                cv=5, scoring=['accuracy', 'precision', 'recall', 'f1'],
                return_train_score=True
            )
            
            # Métricas promedio
            metrics = {
                'accuracy': np.mean(cv_results['test_accuracy']),
                'precision': np.mean(cv_results['test_precision']),
                'recall': np.mean(cv_results['test_recall']),
                'f1': np.mean(cv_results['test_f1'])
            }
            
            detailed_results[name] = metrics
        
        # Añadir gráfico de barras de métricas
        metrics_df = pd.DataFrame(detailed_results).T
        
        for metric in ['accuracy', 'precision', 'recall', 'f1']:
            fig.add_trace(
                go.Bar(x=metrics_df.index, y=metrics_df[metric], name=metric.title()),
                row=1, col=2
            )
        
        # Configurar layout
        fig.update_layout(
            title="Análisis Detallado de Modelos IMDB",
            showlegend=True,
            height=800
        )
        
        fig.show()
        
        return detailed_results
    
    def plot_neural_network_architectures(self, X_train, X_test, y_train, y_test):
        """Comparar diferentes arquitecturas de redes neuronales"""
        
        input_dim = X_train.shape[1]
        
        # Definir arquitecturas
        def create_shallow_network():
            model = Sequential([
                Dense(128, activation='relu', input_shape=(input_dim,)),
                Dropout(0.5),
                Dense(1, activation='sigmoid')
            ])
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            return model
        
        def create_deep_network():
            model = Sequential([
                Dense(512, activation='relu', input_shape=(input_dim,)),
                Dropout(0.5),
                Dense(256, activation='relu'),
                Dropout(0.3),
                Dense(128, activation='relu'),
                Dropout(0.2),
                Dense(1, activation='sigmoid')
            ])
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            return model
        
        def create_wide_network():
            model = Sequential([
                Dense(1024, activation='relu', input_shape=(input_dim,)),
                Dropout(0.6),
                Dense(1024, activation='relu'),
                Dropout(0.4),
                Dense(1, activation='sigmoid')
            ])
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            return model
        
        architectures = {
            'Shallow (128)': create_shallow_network,
            'Deep (512-256-128)': create_deep_network,
            'Wide (1024-1024)': create_wide_network
        }
        
        # Crear subplots para cada arquitectura
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        neural_results = {}
        
        for idx, (name, model_func) in enumerate(architectures.items()):
            print(f"Entrenando arquitectura: {name}")
            
            model = model_func()
            
            # Entrenar con callback para guardar historia
            history = model.fit(
                X_train, y_train,
                validation_data=(X_test, y_test),
                epochs=25,
                batch_size=32,
                verbose=0
            )
            
            # Gráfico de precisión
            epochs = range(1, len(history.history['accuracy']) + 1)
            
            axes[0, idx].plot(epochs, history.history['accuracy'], 'o-', 
                             color='#2E86AB', label='Entrenamiento', linewidth=2)
            axes[0, idx].plot(epochs, history.history['val_accuracy'], 'o-', 
                             color='#A23B72', label='Validación', linewidth=2)
            
            axes[0, idx].set_xlabel('Época')
            axes[0, idx].set_ylabel('Precisión')
            axes[0, idx].set_title(f'Precisión - {name}', fontsize=12, fontweight='bold')
            axes[0, idx].legend()
            axes[0, idx].grid(True, alpha=0.3)
            
            # Gráfico de pérdida
            axes[1, idx].plot(epochs, history.history['loss'], 'o-', 
                             color='#FF6B6B', label='Entrenamiento', linewidth=2)
            axes[1, idx].plot(epochs, history.history['val_loss'], 'o-', 
                             color='#4ECDC4', label='Validación', linewidth=2)
            
            axes[1, idx].set_xlabel('Época')
            axes[1, idx].set_ylabel('Pérdida')
            axes[1, idx].set_title(f'Pérdida - {name}', fontsize=12, fontweight='bold')
            axes[1, idx].legend()
            axes[1, idx].grid(True, alpha=0.3)
            
            # Guardar mejores resultados
            best_val_acc = max(history.history['val_accuracy'])
            best_epoch = np.argmax(history.history['val_accuracy']) + 1
            
            neural_results[name] = {
                'best_accuracy': best_val_acc,
                'best_epoch': best_epoch,
                'history': history.history
            }
            
            # Añadir información en el gráfico
            axes[0, idx].text(0.02, 0.95, 
                             f'Mejor: {best_val_acc:.3f}\nÉpoca: {best_epoch}', 
                             transform=axes[0, idx].transAxes, fontsize=10,
                             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))
        
        plt.tight_layout()
        plt.suptitle('Comparación de Arquitecturas de Redes Neuronales', 
                     fontsize=16, fontweight='bold', y=1.02)
        plt.savefig('neural_architectures_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        return neural_results
    
    def create_final_comparison_dashboard(self, traditional_results, neural_results, 
                                        vectorization_results):
        """Dashboard final con todos los resultados"""
        
        # Preparar datos para visualización
        all_scores = {}
        
        # Resultados tradicionales
        for method, vec_result in vectorization_results.items():
            all_scores[f"Traditional-{method}"] = vec_result['final_accuracy']
        
        # Resultados de redes neuronales
        for arch, neural_result in neural_results.items():
            all_scores[f"Neural-{arch}"] = neural_result['best_accuracy']
        
        # Crear dashboard interactivo
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Comparación General de Precisión', 
                           'Top 5 Modelos', 
                           'Distribución de Precisión por Tipo',
                           'Análisis de Overfitting'),
            specs=[[{"type": "bar"}, {"type": "bar"}],
                   [{"type": "box"}, {"type": "scatter"}]]
        )
        
        # Gráfico 1: Comparación general
        models = list(all_scores.keys())
        scores = list(all_scores.values())
        colors = ['blue' if 'Traditional' in m else 'red' for m in models]
        
        fig.add_trace(
            go.Bar(x=models, y=scores, marker_color=colors, 
                   name="Modelos", showlegend=False),
            row=1, col=1
        )
        
        # Gráfico 2: Top 5 modelos
        sorted_models = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        top_models, top_scores = zip(*sorted_models)
        
        fig.add_trace(
            go.Bar(x=list(top_models), y=list(top_scores), 
                   marker_color='green', name="Top 5", showlegend=False),
            row=1, col=2
        )
        
        # Gráfico 3: Distribución por tipo
        traditional_scores = [s for k, s in all_scores.items() if 'Traditional' in k]
        neural_scores = [s for k, s in all_scores.items() if 'Neural' in k]
        
        fig.add_trace(
            go.Box(y=traditional_scores, name="Tradicionales"),
            row=2, col=1
        )
        fig.add_trace(
            go.Box(y=neural_scores, name="Redes Neuronales"),
            row=2, col=1
        )
        
        # Configurar layout
        fig.update_layout(
            title="Dashboard Final - Análisis de Sentimientos IMDB",
            showlegend=True,
            height=800
        )
        
        # Actualizar ejes
        fig.update_xaxes(tickangle=45, row=1, col=1)
        fig.update_xaxes(tickangle=45, row=1, col=2)
        
        fig.show()
        
        # Crear reporte de resultados
        print("\n" + "="*80)
        print("REPORTE FINAL DE RESULTADOS")
        print("="*80)
        
        print(f"\n🏆 MEJOR MODELO GENERAL:")
        best_model, best_score = sorted_models[0]
        print(f"   {best_model}: {best_score:.4f}")
        
        print(f"\n📊 TOP 5 MODELOS:")
        for i, (model, score) in enumerate(sorted_models[:5], 1):
            print(f"   {i}. {model}: {score:.4f}")
        
        print(f"\n📈 ESTADÍSTICAS POR CATEGORÍA:")
        if traditional_scores:
            print(f"   Modelos Tradicionales: μ={np.mean(traditional_scores):.4f}, σ={np.std(traditional_scores):.4f}")
        if neural_scores:
            print(f"   Redes Neuronales: μ={np.mean(neural_scores):.4f}, σ={np.std(neural_scores):.4f}")
        
        return sorted_models

# Función principal para ejecutar análisis completo
def run_complete_imdb_analysis(dataset_path=None, sample_size=5000):
    """Ejecutar análisis completo del dataset IMDB"""
    
    print("Iniciando análisis completo de sentimientos IMDB...")
    print("=" * 60)
    
    # Inicializar analizador
    analyzer = IMDBSentimentAnalysisVisualizer(dataset_path)
    
    # 1. Cargar y preprocesar datos
    print("\n1. Cargando y preprocesando datos...")
    texts, labels = analyzer.load_and_preprocess_data(sample_size)
    print(f"   Datos cargados: {len(texts)} muestras")
    print(f"   Distribución: {np.sum(labels)} positivas, {len(labels) - np.sum(labels)} negativas")
    
    # 2. Preprocesamiento con SpaCy
    print("\n2. Preprocesando con SpaCy...")
    processed_texts = analyzer.preprocess_with_spacy(texts)
    
    # 3. Comparación de vectorización
    print("\n3. Comparando métodos de vectorización...")
    vectorization_results, X_train, X_test, y_train, y_test = analyzer.create_vectorization_comparison(
        processed_texts, labels
    )
    
    # 4. Análisis detallado de modelos
    print("\n4. Análisis detallado de modelos tradicionales...")
    traditional_results = analyzer.detailed_model_analysis(X_train, X_test, y_train, y_test)
    
    # 5. Análisis de redes neuronales
    print("\n5. Comparando arquitecturas de redes neuronales...")
    neural_results = analyzer.plot_neural_network_architectures(X_train, X_test, y_train, y_test)
    
    # 6. Dashboard final
    print("\n6. Creando dashboard final...")
    final_rankings = analyzer.create_final_comparison_dashboard(
        traditional_results, neural_results, vectorization_results
    )
    
    print("\n" + "=" * 60)
    print("Análisis completado. Archivos generados:")
    print("- vectorization_comparison.png")
    print("- neural_architectures_comparison.png")
    print("- Dashboard interactivo en navegador")
    
    return {
        'vectorization_results': vectorization_results,
        'traditional_results': traditional_results,
        'neural_results': neural_results,
        'final_rankings': final_rankings
    }

# Ejemplo de uso
if __name__ == "__main__":
    # Ejecutar análisis completo
    # Para usar con datos reales, especifica la ruta del archivo CSV:
    # results = run_complete_imdb_analysis("path/to/imdb_dataset.csv", sample_size=10000)
    
    # Para ejecutar con datos sintéticos:
    results = run_complete_imdb_analysis(sample_size=3000)
    
    print(f"\nAnálisis completado exitosamente!")
    print(f"Mejor modelo: {results['final_rankings'][0][0]} con {results['final_rankings'][0][1]:.4f} de precisión")


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\Diego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ------- -------------------------------- 2.9/16.3 MB 13.9 MB/s eta 0:00:01
   ------------ --------------------------- 5.2/16.3 MB 12.7 MB/s eta 0:00:01
   ---------------------- ----------------- 9.2/16.3 MB 14.6 MB/s eta 0:00:01
   ------------------------------- -------- 12.8/16.3 MB 15.5 MB/s eta 0:00:01
   ---------------------------------------  16.3/16.3 MB 15.7 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 14.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Iniciando análisis completo de sentimientos IMDB...

1. Cargando y preprocesando datos...
Usando dataset de ejemplo (20newsgroups adaptado)...


KeyboardInterrupt: 